<a href="https://colab.research.google.com/github/KirtanaPrem/conformaldock/blob/main/ConformalDock_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1 — Install dependencies
!pip install rdkit scikit-learn pandas numpy requests -q
print("Done installing!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 37.5 MB/s eta 0:00:00
Done installing!


In [2]:
# Cell 2 — Fetch real binding data from ChEMBL (expanded targets)
import requests
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

def get_features(mol):
    return [
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        rdMolDescriptors.CalcNumHBD(mol),
        rdMolDescriptors.CalcNumHBA(mol),
        rdMolDescriptors.CalcNumRings(mol),
        rdMolDescriptors.CalcNumRotatableBonds(mol),
        Descriptors.TPSA(mol),
        mol.GetNumAtoms(),
        Descriptors.NumAromaticRings(mol),
        Descriptors.FractionCSP3(mol),
        Descriptors.NumHeteroatoms(mol),
        Descriptors.RingCount(mol),
        Descriptors.NumRadicalElectrons(mol),
        Descriptors.NumValenceElectrons(mol),
        rdMolDescriptors.CalcNumAmideBonds(mol),
        rdMolDescriptors.CalcNumHeterocycles(mol),
    ]

# 50 diverse ChEMBL targets covering different protein families
TARGETS = [
    "CHEMBL202", "CHEMBL204", "CHEMBL1827", "CHEMBL261", "CHEMBL340",
    "CHEMBL205", "CHEMBL1862", "CHEMBL3784", "CHEMBL279", "CHEMBL264",
    "CHEMBL301", "CHEMBL2111367", "CHEMBL325", "CHEMBL4523", "CHEMBL217",
    "CHEMBL218", "CHEMBL220", "CHEMBL224", "CHEMBL228", "CHEMBL230",
    "CHEMBL233", "CHEMBL234", "CHEMBL235", "CHEMBL237", "CHEMBL238",
    "CHEMBL240", "CHEMBL244", "CHEMBL245", "CHEMBL246", "CHEMBL247",
    "CHEMBL251", "CHEMBL252", "CHEMBL253", "CHEMBL255", "CHEMBL256",
    "CHEMBL257", "CHEMBL258", "CHEMBL259", "CHEMBL260", "CHEMBL262",
    "CHEMBL263", "CHEMBL265", "CHEMBL266", "CHEMBL267", "CHEMBL268",
    "CHEMBL269", "CHEMBL270", "CHEMBL271", "CHEMBL272", "CHEMBL273",
]

all_smiles = []
all_scores = []

print("Fetching data from", len(TARGETS), "ChEMBL targets...")

for i, target_id in enumerate(TARGETS):
    try:
        url = (
            "https://www.ebi.ac.uk/chembl/api/data/activity"
            "?target_chembl_id=" + target_id +
            "&standard_type=IC50&standard_relation=%3D"
            "&assay_type=B&pchembl_value__isnull=false"
            "&limit=100&format=json"
        )
        resp = requests.get(url, timeout=15)
        if resp.status_code != 200:
            continue
        for act in resp.json().get("activities", []):
            smiles = act.get("canonical_smiles")
            pchembl = act.get("pchembl_value")
            if not smiles or not pchembl:
                continue
            try:
                score = -(float(pchembl) * 1.364)
            except:
                continue
            if not (-15 < score < -2):
                continue
            mol = Chem.MolFromSmiles(smiles)
            if mol and mol.GetNumAtoms() <= 80:
                all_smiles.append(smiles)
                all_scores.append(score)
    except:
        continue
    if (i + 1) % 10 == 0:
        print(f"  Processed {i+1}/{len(TARGETS)} targets — {len(all_smiles)} molecules so far")

print(f"\nTotal molecules collected: {len(all_smiles)}")
print(f"Score range: {min(all_scores):.2f} to {max(all_scores):.2f} kcal/mol")

Fetching data from 50 ChEMBL targets...
  Processed 10/50 targets — 990 molecules so far
  Processed 20/50 targets — 1990 molecules so far
  Processed 30/50 targets — 2965 molecules so far
  Processed 40/50 targets — 3931 molecules so far
  Processed 50/50 targets — 4629 molecules so far

Total molecules collected: 4629
Score range: -14.59 to -4.80 kcal/mol


In [3]:
# Cell 3 — Train model and run conformal validation
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

print("Building feature matrix...")
X_all = []
y_all = []

for smiles, score in zip(all_smiles, all_scores):
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        X_all.append(get_features(mol))
        y_all.append(score)

X_all = np.array(X_all)
y_all = np.array(y_all)
print(f"Feature matrix: {X_all.shape[0]} molecules x {X_all.shape[1]} features")

# Split: 60% train, 20% calibration, 20% test
np.random.seed(42)
n = len(y_all)
idx = np.random.permutation(n)

train_end = int(n * 0.60)
cal_end = int(n * 0.80)

train_idx = idx[:train_end]
cal_idx = idx[train_end:cal_end]
test_idx = idx[cal_end:]

X_train = X_all[train_idx]
X_cal = X_all[cal_idx]
X_test = X_all[test_idx]
y_train = y_all[train_idx]
y_cal = y_all[cal_idx]
y_test = y_all[test_idx]

print(f"Train: {len(y_train)} | Calibration: {len(y_cal)} | Test: {len(y_test)}")

# Scale features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_cal_s = scaler.transform(X_cal)
X_test_s = scaler.transform(X_test)

# Train model
print("\nTraining Gradient Boosting model...")
model = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    random_state=42,
    subsample=0.8
)
model.fit(X_train_s, y_train)
print("Model trained!")

# Calibration — compute residuals
cal_preds = model.predict(X_cal_s)
cal_residuals = np.abs(y_cal - cal_preds)
print(f"\nCalibration residuals — mean: {cal_residuals.mean():.3f}, median: {np.median(cal_residuals):.3f}")

# Run conformal prediction on test set
print("\n=== CONFORMAL PREDICTION VALIDATION ===")
print(f"Test set size: {len(y_test)} molecules\n")

results = []
for coverage_target in [80, 85, 90, 95]:
    alpha = 1.0 - coverage_target / 100
    n_cal = len(cal_residuals)
    level = min(np.ceil((n_cal + 1) * (1 - alpha)) / n_cal, 1.0)
    q_hat = float(np.quantile(cal_residuals, level))

    test_preds = model.predict(X_test_s)
    lower = test_preds - q_hat
    upper = test_preds + q_hat

    covered = np.mean((y_test >= lower) & (y_test <= upper))
    width = 2 * q_hat

    results.append({
        "Coverage requested": str(coverage_target) + "%",
        "Empirical coverage": str(round(covered * 100, 1)) + "%",
        "q_hat (kcal/mol)": round(q_hat, 3),
        "Interval width (kcal/mol)": round(width, 3),
        "Calibrated?": "YES" if covered >= (coverage_target / 100 - 0.02) else "NO"
    })

    print(f"Target {coverage_target}% → Empirical: {covered*100:.1f}% | Width: {width:.2f} kcal/mol | q_hat: {q_hat:.3f}")

print("\n")
df = pd.DataFrame(results)
print(df.to_string(index=False))
print("\n=== VALIDATION COMPLETE ===")

Building feature matrix...
Feature matrix: 4629 molecules x 16 features
Train: 2777 | Calibration: 926 | Test: 926

Training Gradient Boosting model...
Model trained!

Calibration residuals — mean: 1.143, median: 0.953

=== CONFORMAL PREDICTION VALIDATION ===
Test set size: 926 molecules

Target 80% → Empirical: 80.1% | Width: 3.73 kcal/mol | q_hat: 1.864
Target 85% → Empirical: 85.2% | Width: 4.23 kcal/mol | q_hat: 2.113
Target 90% → Empirical: 90.6% | Width: 4.88 kcal/mol | q_hat: 2.442
Target 95% → Empirical: 95.4% | Width: 5.80 kcal/mol | q_hat: 2.899


Coverage requested Empirical coverage  q_hat (kcal/mol)  Interval width (kcal/mol) Calibrated?
               80%              80.1%             1.864                      3.728         YES
               85%              85.2%             2.113                      4.227         YES
               90%              90.6%             2.442                      4.884         YES
               95%              95.4%             2.899 

In [4]:
# Cell 4 — Save results summary
print("=== CONFORMALDOCK VALIDATION RESULTS ===")
print()
print("Dataset: ChEMBL experimental IC50 measurements")
print(f"Total molecules: {len(y_all)}")
print(f"Training set: {len(y_train)} molecules")
print(f"Calibration set: {len(y_cal)} molecules")
print(f"Test set: {len(y_test)} molecules (held-out, never seen during training)")
print()
print("COVERAGE RESULTS:")
print("  80% target → 80.1% empirical ✓")
print("  85% target → 85.2% empirical ✓")
print("  90% target → 90.6% empirical ✓")
print("  95% target → 95.4% empirical ✓")
print()
print("CONCLUSION: Conformal prediction guarantee is valid.")
print("The model is well-calibrated across all coverage levels.")
print()
print("These results will be reported in:")
print("Premnath K. (2026). ConformalDock: Calibrated Uncertainty")
print("Quantification for Molecular Docking Scores using Conformal")
print("Prediction. MSc Bioinformatics and Data Science.")
print("conformaldock-kirtana.streamlit.app")

=== CONFORMALDOCK VALIDATION RESULTS ===

Dataset: ChEMBL experimental IC50 measurements
Total molecules: 4629
Training set: 2777 molecules
Calibration set: 926 molecules
Test set: 926 molecules (held-out, never seen during training)

COVERAGE RESULTS:
  80% target → 80.1% empirical ✓
  85% target → 85.2% empirical ✓
  90% target → 90.6% empirical ✓
  95% target → 95.4% empirical ✓

CONCLUSION: Conformal prediction guarantee is valid.
The model is well-calibrated across all coverage levels.

These results will be reported in:
Premnath K. (2026). ConformalDock: Calibrated Uncertainty
Quantification for Molecular Docking Scores using Conformal
Prediction. MSc Bioinformatics and Data Science.
conformaldock-kirtana.streamlit.app
